In [113]:
pip install "protobuf>=6.31.1,<7.0.0"

Note: you may need to restart the kernel to use updated packages.


In [114]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('yoga')
#Clase Cat será la clase 0.0
#Clase Dog será la clase 1.0

photos =  []
labels = []

In [115]:
#Importamos las imágenes. Hemos metido un try and catch porque hay dos archivos corruptos que dan error al cargarlos.
#Por razones de memoria cojo solo 5000 imágenes de cada clase.
for idx,folder in enumerate(folders):
    for file in listdir('yoga/'+folder):
        #Cargamos la imagen.
        #print(len(listdir('PetImages/'+folder)))
        try:    
            photo = load_img('yoga/'+folder+'/' + file)
            #Convertimos la imagen a un array y la guardamos en la lista.
            photos.append(img_to_array(photo))
            #También guardamos la etiqueta.
            labels.append(float(idx))
            #Usamos del para borrar los datos temporales para no ocupar memoria que vamos a necesitar.
            del photo
        except:
            print('Error en la foto ' + file + ' de la carpeta ' + folder)

In [116]:
#Ponemos todas las imágenes como un numpy array de dos dimensiones.
photos_array = asarray(photos)
y = asarray(labels)
del photos

In [117]:
print(photos_array.shape)

(570, 16, 16, 3)


In [118]:
#Aplanamos las fotos para hacer un random forest.
photos_reshape = photos_array.reshape(photos_array.shape[0],-1)
photos_reshape.shape

(570, 768)

In [119]:
#Escalamos los datos con StandardScarler. También podríamos dividir los datos entre 255.0
from sklearn.preprocessing import StandardScaler
X = photos_reshape
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [120]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [121]:
#RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train,y_train)
from sklearn.metrics import accuracy_score
y_pred = rnd_clf.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.9298245614035088


In [126]:
from tensorflow import keras
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(512,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(128,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(38,activation='softmax',kernel_initializer='glorot_normal'))

In [129]:
model.compile(loss='sparse_categorical_crossentropy', optimizer = keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=1000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=32)

Epoch 1/1000


ValueError: Argument `output` must have rank (ndim) `target.ndim - 1`. Received: target.shape=(None,), output.shape=(None, 16, 16, 38)

In [122]:
#RED NEURONAL CONVOLUCIONAL
#Como para la red convolucional necesito los datos en forma de matriz, hago un reshape de los datos que usé antes (aplanados) 
# con la forma que quiero que tengan.
X = photos_reshape.reshape(570, 16, 16, 3)
X = X/255.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [123]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
from tensorflow import keras

model = keras.models.Sequential()
model.add(Conv2D(16, (3,3), activation='relu', input_shape=(16, 16, 3)))
# 14x14 -> (Pool 2x2) -> 7x7
model.add(MaxPool2D(2,2))

# Bloque 2: 7x7 -> (Conv 3x3) -> 5x5
model.add(Conv2D(32, (3,3), activation='relu'))
# 5x5 -> (Pool 2x2) -> 2x2
model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(250,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(50,activation='softmax',kernel_initializer='glorot_normal'))

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [124]:
#Usamos Nadam. Se podrían usar otros optimizadores, pero el resultado es similar.
model.compile(loss='sparse_categorical_crossentropy', optimizer = keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=32)

Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0217 - loss: 3.8732 - val_accuracy: 0.0192 - val_loss: 3.8185
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0629 - loss: 3.7053 - val_accuracy: 0.0000e+00 - val_loss: 3.6077
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0759 - loss: 3.4667 - val_accuracy: 0.0769 - val_loss: 3.2718
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1171 - loss: 3.1340 - val_accuracy: 0.1538 - val_loss: 2.9001
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1887 - loss: 2.7718 - val_accuracy: 0.1731 - val_loss: 2.5293
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2516 - loss: 2.4598 - val_accuracy: 0.1923 - val_loss: 2.3538
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3037 - loss: 2.1764 - val_accuracy: 0.3462 - val_loss: 2.0716
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3883 - loss: 1.9644 - val_accuracy: 0.365

In [125]:
model.evaluate(X_test,y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9474 - loss: 0.4769


[0.4769402742385864, 0.9473684430122375]